In [ ]:
from tensorflow import keras
from sklearn.model_selection import train_test_split

(train_input, train_target), (test_input, test_target) = \
    keras.datasets.fashion_mnist.load_data()

train_scaled = train_input.reshape(-1, 28, 28, 1)        # (Dimension, Row, Column, Channel)        # 이전 신경망 모델에서는 숫자로 했지만 여기선 색도 포함시키게 할 거. 흑백 = 1

train_scaled, val_scaled, train_target, val_target = \
    train_test_split(
        train_scaled,
        train_target,
        test_size=0.2,
        random_state=42
    )

#### 합성곱(CNN) 층 만들기

In [ ]:
model = keras.Sequential()
model.add(
    keras.layers.Conv2D(
        32,
        kernel_size=3,  # 보통 정사각형으로 구성 (3,3)
        activation='relu',
        padding='same',  # 원본 이미지 크기과 같게  <=> valid
        input_shape=(28, 28, 1)
    )           # Conv2D(필터 갯수, 필터의 구성, 활성화 함수, padding, 형태)
)

c:\Users\tjoeun\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


#### 풀링층

In [4]:
model.add(
    keras.layers.MaxPooling2D(2)    # 2*2 max pooling
)

#### 합성곱층

In [5]:
model.add(
    keras.layers.Conv2D(
        64,     # => 바뀜
        kernel_size=3,  # 보통 정사각형으로 구성 (3,3)
        activation='relu',
        padding='same',  # 원본 이미지 크기과 같게  <=> valid
    )
)
model.add(
    keras.layers.MaxPooling2D(2)    # 2*2 max pooling
)

#### Dense 층

In [6]:
# 입력층
model.add(
    keras.layers.Flatten()  # 평평하게 1차원으로 바꿔라
)

# 은닉층
model.add(
    keras.layers.Dense(100, activation='relu')
)

# Drop out 층
model.add(
    keras.layers.Dropout(0.5)   # 0.5 쳐내. train 잘 맞추는데 valid 못 맞출 때. 원래 출력 해보고 나서 조절하면 됨.
)

# 출력층
model.add(
    keras.layers.Dense(10, activation='softmax')
)

In [7]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 100)            │       313,700 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 333,526 (1.27 MB)

 Trainable params: 333,526 (1.27 MB)

 Non-trainable params: 0 (0.00 B)

#### Model Compile

In [8]:
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

checkpoint_cb = keras.callbacks.ModelCheckpoint('../Data/best_cnn_model.keras')
early_stopping_cb = keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)

#### Training

In [9]:
history = model.fit(
    train_scaled,
    train_target,
    epochs=20,
    validation_data=(val_scaled, val_target),
    callbacks=[checkpoint_cb, early_stopping_cb]
)

Epoch 1/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - accuracy: 0.7546 - loss: 0.8156 - val_accuracy: 0.8527 - val_loss: 0.3927
Epoch 2/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.8372 - loss: 0.4538 - val_accuracy: 0.8814 - val_loss: 0.3254
Epoch 3/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.8578 - loss: 0.3968 - val_accuracy: 0.8875 - val_loss: 0.3060
Epoch 4/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - accuracy: 0.8687 - loss: 0.3588 - val_accuracy: 0.8939 - val_loss: 0.2926
Epoch 5/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - accuracy: 0.8790 - loss: 0.3323 - val_accuracy: 0.8969 - val_loss: 0.2801
Epoch 6/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - accuracy: 0.8833 - loss: 0.3147 - val_accuracy: 0.8959 - val_loss: 0.2757
Epoch 7/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.8909 - loss: 0.3002 - val_accuracy: 0.9022 - val_loss: 0.2724
Epoch 8/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.8938 - loss: 0

#### 평가와 예측